# V3 Grid Search Evaluation

This notebook gives a comparison view of the v3 grid-search results.

What it does:
- ranks RL runs using the selected split metrics
- shows the matching validation and test metrics for the selected runs
- compares the selected RL runs against a few split baselines
- uses bar charts instead of scatter plots so the comparisons are easier to read


## 1. Config

Keep all of the main choices here so the rest of the notebook stays simple.


In [ ]:
from pathlib import Path
import csv
import json

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

RESULTS_FILE_CANDIDATES = [
    "grid_search_runs.csv",
    "grid_search_runs_original_features.csv",
]
DEFAULT_FEATURE_BATCH = "original"


def resolve_notebook_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "machine_learning" / "RL" / "two_opt_chain_rl_v3",
    ]
    for candidate in candidates:
        if not candidate.is_dir():
            continue
        if not (candidate / "splits").exists():
            continue
        if any((candidate / name).exists() for name in RESULTS_FILE_CANDIDATES):
            return candidate
    raise FileNotFoundError(
        "Could not locate the v3 grid-search notebook directory. "
        "Checked the current working directory and the repo-root fallback."
    )


NOTEBOOK_DIR = resolve_notebook_dir()
RESULTS_CSV = next(
    NOTEBOOK_DIR / name
    for name in RESULTS_FILE_CANDIDATES
    if (NOTEBOOK_DIR / name).exists()
)

# Easy switch: choose which split drives ranking, baselines, and plots.
# Options: "validation" or "test"
EVAL_SPLIT = "validation"
if EVAL_SPLIT not in {"validation", "test"}:
    raise ValueError("EVAL_SPLIT must be 'validation' or 'test'.")

SPLIT_CSV = NOTEBOOK_DIR / "splits" / f"{EVAL_SPLIT}.csv"

TOP_K = 5
INCLUDE_FIXED_CHAIN_BASELINE = True
FIXED_CHAIN_NAME = "tket__wisq_bqskit"

# These weights are only used for the mixed ranking categories.
# Keep them here so you can easily change them later.
MIXED_WEIGHTS = {
    "optimization_runtime": {"two_qubit": 0.5, "runtime": 0.5},
    "optimization_depth": {"two_qubit": 0.8, "depth": 0.2},
    "optimization_depth_runtime": {"two_qubit": 0.6, "depth": 0.3, "runtime": 0.1},
}

# Human-friendly category names for tables and plots.
CATEGORY_LABELS = {
    "optimization_only": "Optimization only",
    "depth_only": "Depth only",
    "optimization_runtime": "Optimization + runtime",
    "optimization_depth": "Optimization + depth",
    "optimization_depth_runtime": "Optimization + depth + runtime",
}

SELECTED_UTILITY_COL = f"{EVAL_SPLIT}_utility"
SELECTED_2Q_COL = f"{EVAL_SPLIT}_avg_two_qubit_gate_reduction_pct"
SELECTED_DEPTH_COL = f"{EVAL_SPLIT}_avg_depth_reduction_pct"
SELECTED_RUNTIME_COL = f"{EVAL_SPLIT}_avg_runtime_seconds"
SELECTED_RUNTIME_SCORE_COL = f"{EVAL_SPLIT}_avg_runtime_score"

LEGACY_SWEEP_COLUMNS = [
    "sweep_id",
    "run_index",
    "total_runs",
    "status",
    "algorithm_name",
    "algorithm_seed",
    "reward_type",
    "runtime_weight",
    "final_reward_weight",
    "total_timesteps",
    "grid_search_timesteps",
    "grid_search_candidates",
    "elapsed_seconds",
    "model_path",
    "best_config",
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
    "error_message",
]

WEIGHTED_SWEEP_COLUMNS = [
    "sweep_id",
    "run_index",
    "total_runs",
    "status",
    "algorithm_name",
    "algorithm_seed",
    "reward_type",
    "runtime_weight",
    "two_qubit_weight",
    "depth_weight",
    "final_reward_weight",
    "total_timesteps",
    "grid_search_timesteps",
    "grid_search_candidates",
    "elapsed_seconds",
    "model_path",
    "best_config",
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
    "error_message",
]

CURRENT_SWEEP_COLUMNS = [
    "sweep_id",
    "run_index",
    "total_runs",
    "status",
    "algorithm_name",
    "algorithm_seed",
    "feature_batch",
    "reward_type",
    "runtime_weight",
    "two_qubit_weight",
    "depth_weight",
    "final_reward_weight",
    "total_timesteps",
    "grid_search_timesteps",
    "grid_search_candidates",
    "elapsed_seconds",
    "model_path",
    "best_config",
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
    "error_message",
]

DISPLAY_COLUMNS = [
    "sweep_id",
    "run_index",
    "algorithm_name",
    "algorithm_seed",
    "feature_batch",
    "reward_type",
    "runtime_weight",
    "two_qubit_weight",
    "depth_weight",
    "final_reward_weight",
    "category_score",
    "validation_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_utility",
    "test_avg_two_qubit_gate_reduction_pct",
    "test_avg_depth_reduction_pct",
    "test_avg_runtime_seconds",
    "model_path",
    "best_config",
]

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Results file: {RESULTS_CSV}")
print(f"Evaluation split: {EVAL_SPLIT}")
print(f"Split file: {SPLIT_CSV}")


## 2. Load and clean the RL sweep results

This section keeps only successful runs and converts the important numeric columns.


In [ ]:
SCHEMA_BY_LENGTH = {
    len(LEGACY_SWEEP_COLUMNS): LEGACY_SWEEP_COLUMNS,
    len(WEIGHTED_SWEEP_COLUMNS): WEIGHTED_SWEEP_COLUMNS,
    len(CURRENT_SWEEP_COLUMNS): CURRENT_SWEEP_COLUMNS,
}

SCHEMA_DEFAULTS = {
    "algorithm_seed": 42,
    "feature_batch": DEFAULT_FEATURE_BATCH,
    "two_qubit_weight": 0.5,
    "depth_weight": 0.5,
}


def load_mixed_grid_search_csv(csv_path):
    rows = []
    skipped_lengths = {}

    with open(csv_path, newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader, None)

        for row in reader:
            if not row:
                continue

            columns = SCHEMA_BY_LENGTH.get(len(row))
            if columns is None:
                skipped_lengths[len(row)] = skipped_lengths.get(len(row), 0) + 1
                continue

            parsed = dict(zip(columns, row))
            for key, value in SCHEMA_DEFAULTS.items():
                parsed.setdefault(key, value)
                if parsed[key] in {None, ""}:
                    parsed[key] = value
            rows.append(parsed)

    frame = pd.DataFrame(rows)
    return frame, skipped_lengths


results, skipped_lengths = load_mixed_grid_search_csv(RESULTS_CSV)

if skipped_lengths:
    print(f"Skipped malformed rows by field count: {skipped_lengths}")

# We only want completed runs in the ranking.
results = results[results["status"] == "success"].copy()

numeric_columns = [
    "runtime_weight",
    "two_qubit_weight",
    "depth_weight",
    "final_reward_weight",
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
]
for column in numeric_columns:
    if column in results.columns:
        results[column] = pd.to_numeric(results[column], errors="coerce")

if "best_config" in results.columns:
    results["best_config"] = results["best_config"].fillna("")
    results["best_config"] = results["best_config"].apply(
        lambda text: json.dumps(json.loads(text), sort_keys=True) if text else ""
    )

if "algorithm_seed" not in results.columns:
    results["algorithm_seed"] = 42
results["algorithm_seed"] = pd.to_numeric(results["algorithm_seed"], errors="coerce").fillna(42).astype(int)

if "feature_batch" not in results.columns:
    results["feature_batch"] = DEFAULT_FEATURE_BATCH
results["algorithm_seed"] = pd.to_numeric(results["algorithm_seed"], errors="coerce").fillna(42).astype(int)
results["feature_batch"] = results["feature_batch"].fillna(DEFAULT_FEATURE_BATCH)

required_columns = [SELECTED_2Q_COL, SELECTED_DEPTH_COL, SELECTED_RUNTIME_COL]
missing_required = [column for column in required_columns if column not in results.columns]
if missing_required:
    raise ValueError(f"Missing required result columns: {missing_required}")

results = results.dropna(subset=required_columns).copy()

if results.empty:
    raise ValueError(f"No successful runs were found in {RESULTS_CSV.name}")

summary = pd.DataFrame(
    {
        "successful_runs": [len(results)],
        "algorithms": [results["algorithm_name"].nunique()],
        "feature_batches": [results["feature_batch"].nunique()],
        "reward_types": [results["reward_type"].nunique()],
        "evaluation_split": [EVAL_SPLIT],
    }
)

display(summary)
display(
    results[["feature_batch", "algorithm_name", "reward_type"]]
    .value_counts()
    .rename("run_count")
    .reset_index()
)


## 3. Build validation-based ranking scores

The ranking uses min-max normalization across successful RL runs.

- higher 2Q reduction = better
- higher depth reduction = better
- lower runtime = better


In [ ]:
def compute_metric_bounds(frame):
    return {
        "two_qubit": (
            pd.to_numeric(frame[SELECTED_2Q_COL], errors="coerce").min(),
            pd.to_numeric(frame[SELECTED_2Q_COL], errors="coerce").max(),
        ),
        "depth": (
            pd.to_numeric(frame[SELECTED_DEPTH_COL], errors="coerce").min(),
            pd.to_numeric(frame[SELECTED_DEPTH_COL], errors="coerce").max(),
        ),
        "runtime": (
            pd.to_numeric(frame[SELECTED_RUNTIME_COL], errors="coerce").min(),
            pd.to_numeric(frame[SELECTED_RUNTIME_COL], errors="coerce").max(),
        ),
    }


def min_max_higher_is_better(series, min_value=None, max_value=None, clip=True):
    series = pd.to_numeric(series, errors="coerce")
    if min_value is None:
        min_value = series.min()
    if max_value is None:
        max_value = series.max()
    if pd.isna(min_value) or pd.isna(max_value):
        return pd.Series(np.nan, index=series.index)
    if max_value == min_value:
        return pd.Series(1.0, index=series.index)
    normalized = (series - min_value) / (max_value - min_value)
    if clip:
        normalized = normalized.clip(0.0, 1.0)
    return normalized


def min_max_lower_is_better(series, min_value=None, max_value=None, clip=True):
    return 1.0 - min_max_higher_is_better(
        series,
        min_value=min_value,
        max_value=max_value,
        clip=clip,
    )


def add_normalized_columns(frame, metric_bounds, clip=True):
    scored = frame.copy()
    scored["norm_selected_2q"] = min_max_higher_is_better(
        scored[SELECTED_2Q_COL],
        min_value=metric_bounds["two_qubit"][0],
        max_value=metric_bounds["two_qubit"][1],
        clip=clip,
    )
    scored["norm_selected_depth"] = min_max_higher_is_better(
        scored[SELECTED_DEPTH_COL],
        min_value=metric_bounds["depth"][0],
        max_value=metric_bounds["depth"][1],
        clip=clip,
    )
    scored["norm_selected_runtime"] = min_max_lower_is_better(
        scored[SELECTED_RUNTIME_COL],
        min_value=metric_bounds["runtime"][0],
        max_value=metric_bounds["runtime"][1],
        clip=clip,
    )
    return scored


def add_score_columns(frame):
    scored = frame.copy()
    scored["score_optimization_only"] = scored["norm_selected_2q"]
    scored["score_depth_only"] = scored["norm_selected_depth"]
    scored["score_optimization_runtime"] = (
        MIXED_WEIGHTS["optimization_runtime"]["two_qubit"] * scored["norm_selected_2q"]
        + MIXED_WEIGHTS["optimization_runtime"]["runtime"] * scored["norm_selected_runtime"]
    )
    scored["score_optimization_depth"] = (
        MIXED_WEIGHTS["optimization_depth"]["two_qubit"] * scored["norm_selected_2q"]
        + MIXED_WEIGHTS["optimization_depth"]["depth"] * scored["norm_selected_depth"]
    )
    scored["score_optimization_depth_runtime"] = (
        MIXED_WEIGHTS["optimization_depth_runtime"]["two_qubit"] * scored["norm_selected_2q"]
        + MIXED_WEIGHTS["optimization_depth_runtime"]["depth"] * scored["norm_selected_depth"]
        + MIXED_WEIGHTS["optimization_depth_runtime"]["runtime"] * scored["norm_selected_runtime"]
    )
    return scored


metric_bounds = compute_metric_bounds(results)
results = add_score_columns(add_normalized_columns(results, metric_bounds))

CATEGORY_SCORE_COLUMNS = {
    "optimization_only": "score_optimization_only",
    "depth_only": "score_depth_only",
    "optimization_runtime": "score_optimization_runtime",
    "optimization_depth": "score_optimization_depth",
    "optimization_depth_runtime": "score_optimization_depth_runtime",
}

display(
    results[
        [
            "algorithm_name",
            "reward_type",
            "norm_selected_2q",
            "norm_selected_depth",
            "norm_selected_runtime",
        ]
    ].head()
)



## 4. Build baseline metrics from the validation split

These baselines are computed directly from `splits/validation.csv` so they line up with the same selection split used for ranking.


In [ ]:
split_rows = pd.read_csv(SPLIT_CSV)

required_split_columns = [
    "original_circuit",
    "chain_name",
    "total_two_qubit_gate_reduction_pct",
    "depth_reduction_pct",
    "total_runtime_seconds",
    "runtime_score",
]
missing_split_columns = [column for column in required_split_columns if column not in split_rows.columns]
if missing_split_columns:
    raise ValueError(f"Missing required split columns: {missing_split_columns}")

# Rename the split columns into the same names used by the sweep file,
# so the ranking and plotting code stays simple.
split_rows[SELECTED_2Q_COL] = pd.to_numeric(
    split_rows["total_two_qubit_gate_reduction_pct"], errors="coerce"
)
split_rows[SELECTED_DEPTH_COL] = pd.to_numeric(
    split_rows["depth_reduction_pct"], errors="coerce"
)
split_rows[SELECTED_RUNTIME_COL] = pd.to_numeric(
    split_rows["total_runtime_seconds"], errors="coerce"
)
split_rows[SELECTED_RUNTIME_SCORE_COL] = pd.to_numeric(
    split_rows["runtime_score"], errors="coerce"
)

# Keep the RL runs clipped to [0, 1], but let the split rows stay
# unclipped. Otherwise a baseline that beats every RL run gets flattened
# to 1.0, which creates false ties and breaks the per-circuit oracle.
split_rows = add_score_columns(
    add_normalized_columns(split_rows, metric_bounds, clip=False)
)

# Average each chain over the full selected split.
chain_summary = (
    split_rows.groupby("chain_name", as_index=False)
    .agg(
        **{
            SELECTED_2Q_COL: (SELECTED_2Q_COL, "mean"),
            SELECTED_DEPTH_COL: (SELECTED_DEPTH_COL, "mean"),
            SELECTED_RUNTIME_COL: (SELECTED_RUNTIME_COL, "mean"),
            SELECTED_RUNTIME_SCORE_COL: (SELECTED_RUNTIME_SCORE_COL, "mean"),
        }
    )
)
chain_summary = add_score_columns(
    add_normalized_columns(chain_summary, metric_bounds, clip=False)
)


def baseline_row_from_chain(row, baseline_name):
    return {
        "baseline_name": baseline_name,
        "selected_chain": row["chain_name"],
        SELECTED_2Q_COL: row[SELECTED_2Q_COL],
        SELECTED_DEPTH_COL: row[SELECTED_DEPTH_COL],
        SELECTED_RUNTIME_COL: row[SELECTED_RUNTIME_COL],
        SELECTED_RUNTIME_SCORE_COL: row[SELECTED_RUNTIME_SCORE_COL],
    }


def oracle_row(category_key):
    score_column = CATEGORY_SCORE_COLUMNS[category_key]
    chosen = split_rows.loc[
        split_rows.groupby("original_circuit")[score_column].idxmax()
    ].copy()
    return {
        "baseline_name": "Oracle best chain",
        "selected_chain": "per-circuit oracle",
        SELECTED_2Q_COL: chosen[SELECTED_2Q_COL].mean(),
        SELECTED_DEPTH_COL: chosen[SELECTED_DEPTH_COL].mean(),
        SELECTED_RUNTIME_COL: chosen[SELECTED_RUNTIME_COL].mean(),
        SELECTED_RUNTIME_SCORE_COL: chosen[SELECTED_RUNTIME_SCORE_COL].mean(),
    }


baseline_results = {}
for category_key, score_column in CATEGORY_SCORE_COLUMNS.items():
    rows = []

    best_single = chain_summary[~chain_summary["chain_name"].str.contains("__", regex=False)]
    if not best_single.empty:
        rows.append(
            baseline_row_from_chain(
                best_single.sort_values(score_column, ascending=False).iloc[0],
                "Best single optimizer",
            )
        )

    fixed_chains = chain_summary[chain_summary["chain_name"].str.contains("__", regex=False)]
    if not fixed_chains.empty:
        rows.append(
            baseline_row_from_chain(
                fixed_chains.sort_values(score_column, ascending=False).iloc[0],
                "Best overall fixed chain",
            )
        )

    rows.append(oracle_row(category_key))

    if INCLUDE_FIXED_CHAIN_BASELINE and FIXED_CHAIN_NAME in set(chain_summary["chain_name"]):
        fixed_chain_row = chain_summary[chain_summary["chain_name"] == FIXED_CHAIN_NAME].iloc[0]
        rows.append(baseline_row_from_chain(fixed_chain_row, f"Fixed chain ({FIXED_CHAIN_NAME})"))

    baseline_frame = pd.DataFrame(rows)
    baseline_frame = add_score_columns(
        add_normalized_columns(baseline_frame, metric_bounds, clip=False)
    )
    baseline_frame["category_score"] = baseline_frame[score_column]
    baseline_results[category_key] = baseline_frame.sort_values("category_score", ascending=False)

display(baseline_results["optimization_depth_runtime"])



## 5. Rank the top runs in each category

Each table below is ranked using validation only, but also shows the matching test metrics for the selected rows.


In [ ]:
def top_k_table(category_key, top_k=TOP_K):
    score_column = CATEGORY_SCORE_COLUMNS[category_key]
    ranked = results.sort_values(score_column, ascending=False).head(top_k).copy()
    ranked["category_score"] = ranked[score_column]
    display_columns = [column for column in DISPLAY_COLUMNS if column in ranked.columns]
    return ranked[display_columns]


for category_key, category_label in CATEGORY_LABELS.items():
    print(f"\n===== {category_label} ({EVAL_SPLIT.title()} ranking) =====")
    display(top_k_table(category_key))
    print(f"{EVAL_SPLIT.title()} baselines for this category")
    display(
        baseline_results[category_key][
            [
                "baseline_name",
                "selected_chain",
                "category_score",
                SELECTED_2Q_COL,
                SELECTED_DEPTH_COL,
                SELECTED_RUNTIME_COL,
            ]
        ]
    )



## 6. Cleaner comparison plots

Instead of scatter plots, each section below uses bar charts.

For every category, the notebook shows the top 5 RL runs and the baselines together in the same three charts:
- runtime
- 2Q gate reduction
- depth reduction


In [ ]:
def short_run_label(row, rank_number):
    algorithm = str(row["algorithm_name"]).upper()
    reward = str(row["reward_type"]).replace("_", " ")
    return f"RL #{rank_number} | {algorithm} | {reward}"


def format_runtime_weight(value):
    if pd.isna(value):
        return "NA"
    return f"{float(value):.3g}"


def build_top_plot_frame(category_key, top_k=TOP_K):
    score_column = CATEGORY_SCORE_COLUMNS[category_key]
    top_runs = results.sort_values(score_column, ascending=False).head(top_k).copy().reset_index(drop=True)
    top_runs["rank"] = np.arange(1, len(top_runs) + 1)
    top_runs["plot_label"] = top_runs.apply(
        lambda row: short_run_label(row, int(row["rank"])),
        axis=1,
    )
    top_runs["detail_label"] = top_runs.apply(
        lambda row: (
            f"{row['plot_label']}\n"
            f"seed={int(row.get('algorithm_seed', 42))}, "
            f"fb={row.get('feature_batch', DEFAULT_FEATURE_BATCH)}, "
            f"runtime_w={format_runtime_weight(row['runtime_weight'])}, "
            f"2q_w={format_runtime_weight(row['two_qubit_weight'])}, "
            f"depth_w={format_runtime_weight(row['depth_weight'])}, "
            f"final_w={format_runtime_weight(row['final_reward_weight'])}"
        ),
        axis=1,
    )
    top_runs["category_score"] = top_runs[score_column]
    top_runs["entry_type"] = "Top RL run"
    return top_runs


def build_baseline_plot_frame(category_key):
    baseline_frame = baseline_results[category_key].copy().reset_index(drop=True)
    baseline_frame["plot_label"] = baseline_frame["baseline_name"]
    baseline_frame["detail_label"] = baseline_frame["baseline_name"]
    baseline_frame["entry_type"] = "Baseline"
    return baseline_frame


def build_combined_plot_frame(category_key):
    top_runs = build_top_plot_frame(category_key)
    baselines = build_baseline_plot_frame(category_key)
    combined = pd.concat([top_runs, baselines], ignore_index=True, sort=False)

    # Put the RL runs first, then the baselines below them.
    ordered_labels = list(top_runs["detail_label"]) + list(baselines["detail_label"])
    combined["detail_label"] = pd.Categorical(
        combined["detail_label"],
        categories=ordered_labels,
        ordered=True,
    )
    return combined


def add_bar_labels(ax, fmt="{:.3f}"):
    for container in ax.containers:
        labels = []
        for bar in container:
            width = bar.get_width()
            if np.isnan(width):
                labels.append("")
            else:
                labels.append(fmt.format(width))
        ax.bar_label(container, labels=labels, padding=4, fontsize=9)


def plot_metric_comparison(ax, combined, metric_column, title, xlabel, fmt):
    sns.barplot(
        data=combined,
        x=metric_column,
        y="detail_label",
        hue="entry_type",
        dodge=False,
        palette={"Top RL run": "#4C78A8", "Baseline": "#E45756"},
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    add_bar_labels(ax, fmt=fmt)
    handles, labels = ax.get_legend_handles_labels()
    dedup = dict(zip(labels, handles))
    ax.legend(dedup.values(), dedup.keys(), title="Entry type", loc="best")


def plot_category_section(category_key):
    category_label = CATEGORY_LABELS[category_key]
    combined = build_combined_plot_frame(category_key)

    fig, axes = plt.subplots(1, 3, figsize=(24, max(7, 0.7 * len(combined))))

    plot_metric_comparison(
        ax=axes[0],
        combined=combined,
        metric_column=SELECTED_RUNTIME_COL,
        title=f"{category_label}: runtime comparison",
        xlabel=f"{EVAL_SPLIT.title()} avg runtime (s)",
        fmt="{:.2f}",
    )

    plot_metric_comparison(
        ax=axes[1],
        combined=combined,
        metric_column=SELECTED_2Q_COL,
        title=f"{category_label}: 2Q gate reduction comparison",
        xlabel=f"{EVAL_SPLIT.title()} avg 2Q reduction (%)",
        fmt="{:.2f}",
    )

    plot_metric_comparison(
        ax=axes[2],
        combined=combined,
        metric_column=SELECTED_DEPTH_COL,
        title=f"{category_label}: depth reduction comparison",
        xlabel=f"{EVAL_SPLIT.title()} avg depth reduction (%)",
        fmt="{:.2f}",
    )

    fig.suptitle(
        f"{category_label}: top {TOP_K} RL runs vs baselines ({EVAL_SPLIT} split)",
        fontsize=16,
        y=1.02,
    )
    plt.tight_layout()
    plt.show()


for category_key, category_label in CATEGORY_LABELS.items():
    print(f"\n{'=' * 80}\n{category_label} ({EVAL_SPLIT.title()})\n{'=' * 80}")
    plot_category_section(category_key)
